# Spearman Correlation Analysis

This notebook evaluates monotonic relationships between each feature and the binary target loan_status using **Spearman's rank correlation**.

Spearman correlation is useful for detecting **ordinal or non-linear but monotonic** trends a common scenario in credit scoring when input features are not normally distributed or linearly related to the outcome.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
from scipy.stats import rankdata

import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

## Load Dataset

The analysis uses the univariate WOE/IV-prepared dataset, excluding Information Value, to maintain consistency with previous statistical variable assessments.

In [2]:
data = pd.read_csv(r'/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/Internal_test_0_datasetForUnivariateVSExceptIV.csv', index_col=[0])

In [3]:
data.head()

,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
32229,1.002263,63788.0,17,-51.803563,5000.0,33.130018,11.01,0.08,12.0,686,-105.711715,0
11,1.002263,13113.0,0,131.015335,4500.0,-25.457321,8.63,0.34,2.0,651,-105.711715,1
9010,0.369066,59603.0,0,-51.803563,8000.0,11.545480,14.96,0.13,2.0,570,-105.711715,1
7030,-1.125529,54919.0,0,-51.803563,6225.0,55.131699,11.54,0.11,4.0,706,914.813057,0
21143,1.002263,55000.0,7,77.011839,6000.0,11.545480,9.32,0.11,9.0,643,914.813057,0


In [4]:
data.shape

(32400, 12)

In [5]:
lstfeature = list(data.columns)

In [6]:
len(lstfeature)

12

## Prepare Features

We remove the target variable loan_status from the feature list before computing Spearman correlations.

In [7]:
lstfeature.remove('loan_status')

In [8]:
len(lstfeature)

11

In [9]:
variable = []
spearman = []
pvalue = []

## Compute Spearman Correlation

For each feature, we calculate the Spearman correlation coefficient and its p-value with respect to the target. This helps identify monotonic relationships useful in both logistic regression and tree-based models.

In [10]:
for i in lstfeature:
    variable.append(i)
    spearman.append(stats.spearmanr(data[i], data['loan_status']).statistic)
    pvalue.append(stats.spearmanr(data[i], data['loan_status']).pvalue)

In [11]:
dict = {'variable': variable, 'spearman': spearman, 'pvalue': pvalue}
df = pd.DataFrame(dict)

In [12]:
df

,variable,spearman,pvalue
0,person_education,-0.003458,5.337012e-01
1,person_income,-0.265845,0.000000e+00
2,person_emp_exp,-0.025267,5.399741e-06
3,person_home_ownership,-0.255919,0.000000e+00
4,loan_amnt,0.093321,1.377805e-63
5,loan_intent,-0.139045,1.433095e-139
6,loan_int_rate,0.318893,0.000000e+00
7,loan_percent_income,0.321968,0.000000e+00
8,cb_person_cred_hist_length,-0.019365,4.905449e-04
9,credit_score,-0.008791,1.135726e-01


In [13]:
df['spearman'] = df['spearman'].abs()

## Rank and Export

We sort features by absolute Spearman coefficient strength and export the ranked list to CSV for use in feature selection and comparison with Hoeffding's D or XGBoost-based importance scores.

In [14]:
df = df.sort_values(by = ['spearman'], ascending = False).reset_index().drop(columns = ['index'])

## Interpretation of Spearman Correlation Results

This table ranks features based on their **monotonic correlation** with the target variable loan_status using Spearman’s rank method.

- **previous_loan_defaults_on_file** shows the strongest monotonic relationship (ρ ≈ 0.54), reinforcing its value as a top risk signal, consistently ranked highly across IV, XGBoost, and Hoeffding's D.

- Features like **loan_percent_income**, **loan_int_rate**, and **person_income** also show moderate positive Spearman correlations, suggesting that as these values increase, the likelihood of default tends to increase monotonically.

- **credit_score** and **person_education** have very weak (ρ ≈ 0.01 or less) and statistically non-significant correlations (p > 0.05), indicating limited monotonic predictive power.

> **Note**: A low Spearman value does not necessarily mean the variable is useless — it could still be valuable in nonlinear or interaction-heavy models like Random Forest or XGBoost.

This analysis strengthens confidence in high-performing features and helps deprioritize those with weak and non-monotonic relationships.


In [15]:
df

,variable,spearman,pvalue
0,previous_loan_defaults_on_file,0.542635,0.000000e+00
1,loan_percent_income,0.321968,0.000000e+00
2,loan_int_rate,0.318893,0.000000e+00
3,person_income,0.265845,0.000000e+00
4,person_home_ownership,0.255919,0.000000e+00
5,loan_intent,0.139045,1.433095e-139
6,loan_amnt,0.093321,1.377805e-63
7,person_emp_exp,0.025267,5.399741e-06
8,cb_person_cred_hist_length,0.019365,4.905449e-04
9,credit_score,0.008791,1.135726e-01


In [16]:
df.to_csv('/content/drive/MyDrive/Colab Notebooks/Scorecard_Project/spearman.csv')